# CFG: barrido de $w$ y comparativa entre schedules

Dos experimentos en un solo notebook:

1. **Sweep de $w$** sobre VP-Cosine para una clase fija — muestra el trade-off fidelidad/diversidad.
2. **Comparativa de schedules** (Linear, Cosine, Exponential) para la misma clase y $w$ — muestra el efecto del schedule en la calidad de las muestras condicionales.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != 'proyecto_AAIII_02_diffusion_models':
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

from score_model_digits_col import ScoreNet as ScoreNetColor
from diffusion_lib import (
    VPProcess, LinearSchedule, CosineSchedule, ExponentialSchedule,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

ModuleNotFoundError: No module named 'score_model_digits_color'

## 1. Modelo condicional y utilidades CFG

Mismo `ConditionalScoreNetColor` y mismo sampler CFG que el notebook 1, duplicados aquí para que este notebook sea autocontenido.

In [ ]:
class ConditionalScoreNetColor(nn.Module):
    def __init__(self, marginal_prob_std, n_classes=10, embed_dim=256):
        super().__init__()
        self.n_classes  = n_classes
        self.null_token = n_classes
        self.class_embed = nn.Embedding(n_classes + 1, embed_dim)
        self.score_net   = ScoreNetColor(marginal_prob_std=marginal_prob_std,
                                         embed_dim=embed_dim)

    def forward(self, x, t, class_label=None):
        B = x.shape[0]
        if class_label is None:
            class_label = torch.full((B,), self.null_token,
                                     device=x.device, dtype=torch.long)
        c_emb = self.class_embed(class_label)
        return self.score_net(x, t, class_emb=c_emb)


def cfg_score(model, x, t, class_label, cfg_scale):
    B = x.shape[0]
    y_cond   = torch.full((B,), int(class_label), device=x.device, dtype=torch.long)
    s_uncond = model(x, t, class_label=None)
    s_cond   = model(x, t, class_label=y_cond)
    return s_uncond + cfg_scale * (s_cond - s_uncond)


@torch.no_grad()
def cfg_sample(model, process, n_images, class_label, cfg_scale,
               img_shape=(3, 32, 32), n_steps=500, T=0.999, eps=1e-3,
               seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    x = process.prior_sample((n_images, *img_shape), device)
    dt     = (eps - T) / n_steps
    t_vals = torch.linspace(T, eps, n_steps + 1, device=device)
    view_shape = [n_images] + [1] * len(img_shape)
    model.eval()
    for i in range(n_steps):
        t = t_vals[i].expand(n_images)
        score = cfg_score(model, x, t, class_label, cfg_scale)
        drift = process.drift_coefficient(x, t) - process.diffusion_coefficient(t).view(*view_shape)**2 * score
        g_t   = process.diffusion_coefficient(t).view(*view_shape)
        z     = torch.randn_like(x)
        x     = x + drift * dt + g_t * np.sqrt(abs(dt)) * z
    return x

## 2. Helper para cargar un checkpoint según el schedule

In [ ]:
CKPT_DIR = PROJECT_DIR / 'color_digits_cond_checkpoints'

SCHEDULE_OPTIONS = {
    'Linear':      (LinearSchedule(),      'color_digits_VP-Linear.pth'),
    'Cosine':      (CosineSchedule(),      'color_digits_VP-Cosine.pth'),
    'Exponential': (ExponentialSchedule(), 'color_digits_VP-Exponential.pth'),
}

def load_cond_model(schedule_name):
    sched, ckpt_file = SCHEDULE_OPTIONS[schedule_name]
    process = VPProcess(schedule=sched)
    model = ConditionalScoreNetColor(marginal_prob_std=process.sigma_t).to(device)
    model.load_state_dict(torch.load(CKPT_DIR / ckpt_file, map_location=device))
    model.eval()
    return model, process

## 3. Experimento 1: barrido de $w$ sobre VP-Cosine

Para una clase fija (digamos $y=7$), generamos una muestra con la **misma semilla** para cada $w \in \{0, 1, 3, 5, 10\}$. Observa cómo:

- $w=0$ → ignora la clase (pase incondicional).
- $w=1$ → condicional puro.
- $w>1$ → fidelidad amplificada, menos diversidad.
- $w$ muy alto → saturación / artefactos.

In [ ]:
model_cos, process_cos = load_cond_model('Cosine')

TARGET_CLASS = 7
WS = [0.0, 1.0, 3.0, 5.0, 10.0]
SEED = 42

samples_w = []
for w in WS:
    print(f'Sampling con w = {w}...')
    s = cfg_sample(model_cos, process_cos, n_images=1,
                   class_label=TARGET_CLASS, cfg_scale=w, seed=SEED)
    samples_w.append(s.cpu())

fig, axes = plt.subplots(1, len(WS), figsize=(len(WS)*2.4, 2.6))
for i, (w, s) in enumerate(zip(WS, samples_w)):
    img = s[0].permute(1, 2, 0).clamp(0, 1).numpy()
    axes[i].imshow(img)
    axes[i].set_title(f'w = {w}')
    axes[i].axis('off')
fig.suptitle(f'Sweep de CFG scale para la clase y={TARGET_CLASS} — VP-Cosine', y=1.05)
plt.tight_layout()

out_dir = PROJECT_DIR / 'figuras'
out_dir.mkdir(exist_ok=True)
fig.savefig(out_dir / 'cfg_w_sweep_color_digits.pdf', bbox_inches='tight', dpi=200)
plt.show()

## 4. Experimento 2: comparativa entre schedules

Misma clase, mismo $w=3$, misma semilla, pero **distinto schedule**. Sirve para §5.2 del informe (Lineal vs Coseno vs Exponencial sobre VP).

In [ ]:
TARGET_CLASS = 7
W = 3.0
SEED = 42
N_PER_SCHEDULE = 4

all_results = {}
for name in ['Linear', 'Cosine', 'Exponential']:
    print(f'Cargando {name}...')
    model, proc = load_cond_model(name)
    samples = cfg_sample(model, proc, n_images=N_PER_SCHEDULE,
                          class_label=TARGET_CLASS, cfg_scale=W, seed=SEED)
    all_results[name] = samples.cpu()
    del model        # libera memoria de GPU
    torch.cuda.empty_cache() if device.type == 'cuda' else None

fig, axes = plt.subplots(3, N_PER_SCHEDULE, figsize=(N_PER_SCHEDULE*2, 6))
for row, (name, samples) in enumerate(all_results.items()):
    for k in range(N_PER_SCHEDULE):
        img = samples[k].permute(1, 2, 0).clamp(0, 1).numpy()
        axes[row, k].imshow(img)
        axes[row, k].axis('off')
    axes[row, 0].set_ylabel(name, rotation=90, labelpad=15, fontsize=12)
fig.suptitle(f'Comparativa de schedules — clase y={TARGET_CLASS}, w={W}', y=1.0)
plt.tight_layout()
fig.savefig(PROJECT_DIR / 'figuras' / 'cfg_schedule_comparison.pdf', bbox_inches='tight', dpi=200)
plt.show()

## 5. Discusión esperada

- **Coseno**: suele dar las muestras más nítidas en imágenes pequeñas (32×32) porque mantiene la señal hasta más tarde.
- **Lineal**: puede saturar de ruido demasiado pronto.
- **Exponencial**: comportamiento agresivo al final, depende mucho de $\beta_\text{max}$.

Estas observaciones son las que vais a contrastar con FID/IS en §5 del informe.